# 📔 Jurnal Harian & Laporan — Pertemuan 2
**Tanggal**: 4 September 2026  |  **Nama**: M. Arief Abdillah  |  **NIM**: [Isi NIM Kamu]
**AOI Proyek**: Sanggala Utara, Tana Toraja

---

## 📌 TUGAS A & C: Laporan Eksplorasi Data

**1. Pencarian Filter Optimal (Tugas A)**
| Periode | Sensor | Cloud < 20% | Cloud < 30% | Filter Optimal Pilihan |
|---------|--------|-------------|-------------|------------------------|
| **T1: 2015-2016** | Landsat 8 | [Isi angka] scene | [Isi angka] scene | Threshold **30%**, agar jumlah citra mencukupi untuk composite median. |
| **T2: 2023-2024** | Landsat 9 | [Isi angka] scene | [Isi angka] scene | Threshold **20%**, karena data sudah cukup banyak dan bersih. |

**2. Ringkasan Kondisi Wilayah (Tugas C)**
* **Ketersediaan Data:** Jumlah scene T1 dan T2 sudah memadai (di atas 5 scene) untuk membuat citra *composite* yang bebas dari tutupan awan.
* **Kondisi Awan Terbaik:** Pada T1 sebesar [Isi % awan T1]% dan T2 sebesar [Isi % awan T2]%.
* **Kerapatan Vegetasi:** Nilai rata-rata NDVI di Sanggala Utara adalah **[Isi Rata-rata NDVI]**. Nilai ini menunjukkan bahwa wilayah kajian didominasi oleh [Pilih salah satu: vegetasi lebat / campuran lahan terbuka / perairan].
* **Dataset Referensi:** Saya menggunakan dataset publik `ESA/WorldCover/v200/2021` sebagai panduan/referensi untuk memastikan akurasi saat pembuatan titik *training sample*.

---

## 🗺️ TUGAS B: Status Proyek Hari Ini

| Komponen | Status | Catatan |
|----------|--------|---------|
| AOI terdefinisi | ✅ Selesai | Pusat: [Isi Titik Pusat Koordinat dari Script 2.1] |
| Composite T1 dibuat | ✅ Selesai | Tahun 2015-2016 (Landsat 8) |
| Composite T2 dibuat | ✅ Selesai | Tahun 2023-2024 (Landsat 9) |
| Titik training dibuat | ✅ Selesai | 40 titik dari 4 kelas (Hutan, Sawah, Pemukiman, Air) |
| Ekspor ke Drive | ✅ Selesai | File: `Composite_T2_Sanggala_Utara` |

---

## 💡 Evaluasi & Refleksi Teknis

**Yang berhasil hari ini:**
Berhasil menampilkan dan mengekspor citra *composite* Sanggala Utara yang sudah bersih dari awan menggunakan teknik *cloud masking* pada *QA_PIXEL*, serta berhasil membuat data sampel observasi (`FeatureCollection`).

**Keputusan teknis yang diambil:**
* **Sensor:** Menggunakan Landsat 8 (T1) dan Landsat 9 (T2) karena memiliki konsistensi band spektral yang baik untuk analisis *time-series*.
* **Titik Kelas:** Menggunakan 4 kelas utama (Hutan = 1, Sawah = 2, Pemukiman = 3, Air = 4) sebagai basis deteksi tutupan lahan.

**Hal paling menarik dari pertemuan ini:**
Mengetahui bahwa kita bisa menggabungkan (*composite*) puluhan foto yang tertutup awan untuk mendapatkan satu foto satelit wilayah Sanggala Utara yang benar-benar bersih dengan hanya menggunakan algoritma pengambil nilai tengah (*median*).


In [1]:
# ============================================================
!pip install earthengine-api geemap --quiet

import ee
import geemap
import pandas as pd

# Autentikasi dan hubungkan ke project milikmu
ee.Authenticate()
ee.Initialize(project="ma-project-of-indonesian")
print("✅ GEE terhubung — siap digunakan!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.8 MB/s eta 0:00:00
✅ GEE terhubung — siap digunakan!


In [2]:
# ============================================================
# Memanggil data batas yang sudah ada di project-mu
aoi_sanggala = ee.FeatureCollection("projects/ma-project-of-indonesian/assets/Sanggala_Utara")
aoi = aoi_sanggala.geometry()

# Verifikasi: hitung dan tampilkan luas AOI dari server GEE
luas_m2  = aoi.area(maxError=1).getInfo()
luas_km2 = luas_m2 / 1_000_000

print(f"📍 AOI Sanggala Utara berhasil dimuat")
print(f"📐 Luas: {luas_km2:.2f} km²")

📍 AOI Sanggala Utara berhasil dimuat
📐 Luas: 19.02 km²


In [3]:
# ============================================================
def ambil_metadata_koleksi(koleksi, nama):
    jumlah = koleksi.size().getInfo()
    if jumlah == 0:
        print(f"⚠ {nama}: Tidak ada citra tersedia!")
        return None

    citra_pertama = koleksi.sort("CLOUD_COVER").first()
    tgl_terbaik   = citra_pertama.date().format("YYYY-MM-dd").getInfo()
    cloud_terbaik = citra_pertama.get("CLOUD_COVER").getInfo()
    cloud_avg = koleksi.aggregate_mean("CLOUD_COVER").getInfo()

    print(f"📊 {nama}")
    print(f"   Jumlah scene         : {jumlah}")
    print(f"   Citra terbaik tanggal: {tgl_terbaik}")
    print(f"   Cloud cover terbaik  : {cloud_terbaik:.1f}%")
    print(f"   Rata-rata cloud cover: {cloud_avg:.1f}% \n")
    return jumlah

# Koleksi T1 (2015-2016) dan T2 (2023-2024)
koleksi_T1 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
               .filterDate("2015-01-01", "2016-12-31")
               .filterBounds(aoi)
               .filterMetadata("CLOUD_COVER", "less_than", 30))

koleksi_T2 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
               .filterDate("2023-01-01", "2024-12-31")
               .filterBounds(aoi)
               .filterMetadata("CLOUD_COVER", "less_than", 20))

print("=" * 50)
print("LAPORAN KETERSEDIAAN DATA SATELIT SANGGALA UTARA")
print("=" * 50)
n_T1 = ambil_metadata_koleksi(koleksi_T1, "Landsat 8 — T1 (2015-2016)")
n_T2 = ambil_metadata_koleksi(koleksi_T2, "Landsat 9 — T2 (2023-2024)")

LAPORAN KETERSEDIAAN DATA SATELIT SANGGALA UTARA
📊 Landsat 8 — T1 (2015-2016)
   Jumlah scene         : 14
   Citra terbaik tanggal: 2015-09-08
   Cloud cover terbaik  : 5.2%
   Rata-rata cloud cover: 18.2% 

📊 Landsat 9 — T2 (2023-2024)
   Jumlah scene         : 1
   Citra terbaik tanggal: 2023-05-01
   Cloud cover terbaik  : 18.3%
   Rata-rata cloud cover: 18.3% 



In [4]:
# ============================================================
def mask_l8sr(image):
    qa = image.select("QA_PIXEL")
    cloud_shadow = qa.bitwiseAnd(1 << 3).neq(0)
    cloud = qa.bitwiseAnd(1 << 4).neq(0)
    mask = cloud_shadow.Or(cloud).Not()  # Perhatikan: Or dan Not huruf besar di Python!
    return image.updateMask(mask).copyProperties(image, image.propertyNames())

# Buat composite
composite_T1 = koleksi_T1.map(mask_l8sr).median().clip(aoi)
composite_T2 = koleksi_T2.map(mask_l8sr).median().clip(aoi)

# Buat dan atur peta interaktif
Map = geemap.Map()
Map.centerObject(aoi, 13)
vis_true = {"bands": ["SR_B4", "SR_B3", "SR_B2"], "min": 7000, "max": 13000, "gamma": 1.4}

Map.addLayer(composite_T1, vis_true, "T1 (2015-2016)", shown=True)
Map.addLayer(composite_T2, vis_true, "T2 (2023-2024)", shown=False)
Map.addLayer(aoi, {"color": "FF0000", "fillColor": "00000000"}, "Batas Sanggala Utara")

# Tampilkan
Map

Map(center=[-3.067353875895265, 119.91898985086468], controls=(WidgetControl(options=['position', 'transparent…